# Cocoon Realtime API — OmniVoice TTS + Wav2Lip + ngrok

Notebook này biến pipeline cũ thành một service Colab:

- Load OmniVoice một lần để sinh WAV tiếng Việt từ `text/voiceover`.
- Load/setup Wav2Lip một lần, sau đó lipsync từng scene/chunk.
- Mở API bằng Flask + ngrok.
- Endpoint nhận request JSON và trả về video URL.
- Có endpoint streaming `render_job_stream` để trả từng chunk ngay khi từng video scene render xong.

Realtime ở đây là realtime theo chunk: mỗi scene xong thì trả URL video ngay, không phải stream từng frame video.


In [ ]:
# ============================================================
# 1) INSTALL DEPS
# ============================================================
# Colab recommended: Runtime -> Change runtime type -> GPU

!apt-get update -y >/dev/null
!apt-get install -y ffmpeg fonts-noto-core fonts-noto-extra >/dev/null
!fc-cache -fv >/dev/null

# API + TTS deps
!pip install -q omnivoice soundfile pandas tqdm flask flask-cors pyngrok nest_asyncio requests

# Wav2Lip legacy deps are installed after cloning Wav2Lip in the setup cell.
print("Install base deps done.")


In [ ]:
# ============================================================
# 2) CONFIG
# ============================================================
from pathlib import Path
import os

# Drive project root giữ theo base cũ
PROJECT_DIR = Path("/content/drive/MyDrive/cocoon_ai_video")
DATA_DIR = PROJECT_DIR / "data"
MODEL_DIR = PROJECT_DIR / "models"

# Existing assets on Drive
VIDEO_TEMPLATE_DIR = DATA_DIR / "video_templates"
SCENE_IMAGE_DIR = DATA_DIR / "scene_images"
VOICE_DIR = DATA_DIR / "voices"

# Output mới cho API jobs
API_OUTPUT_ROOT = DATA_DIR / "outputs" / "ngrok_realtime_jobs"

# Wav2Lip
WAV2LIP_DIR = Path("/content/Wav2Lip")
CHECKPOINT_SRC = MODEL_DIR / "Wav2Lip-SD-GAN.pt"
CHECKPOINT_DST = WAV2LIP_DIR / "checkpoints" / "Wav2Lip-SD-GAN.pt"
RUNTIME_DIR = Path("/content/wav2lip_runtime")

# OmniVoice
LANGUAGE_ID = "vi"
MODEL_ID = "k2-fsa/OmniVoice"
SAMPLE_RATE = 24000
NUM_STEP = 32
SPEED = 1.0
DEFAULT_DURATION = None
SEED = 42

# Một voice clone cố định cho toàn bộ livestream
# Nếu file chưa tồn tại, cell load model sẽ cho upload bằng Colab.
REF_AUDIO_PATH = VOICE_DIR / "host_ref.wav"
REF_TEXT = None

# Render config
TARGET_W = 1080
TARGET_H = 1920
FPS = 25
CRF = 20                 # 18 đẹp hơn, 20 nhanh hơn/nhẹ hơn
PRESET = "veryfast"
PADS = ["0", "20", "0", "0"]
RESIZE_FACTOR = "1"
WAV2LIP_BATCH_SIZE = "4" # tăng 8 nếu GPU đủ VRAM
ALLOW_STATIC_IMAGE_FOR_LIPSYNC = False
STRICT_LIPSYNC = True

# API / ngrok
PORT = 7860
HOST = "0.0.0.0"

# Có thể paste token ở đây hoặc để trống rồi notebook sẽ hỏi bằng getpass.
# Lấy token ở ngrok dashboard, không commit token lên repo.
NGROK_AUTHTOKEN = os.environ.get("NGROK_AUTHTOKEN", "")

for p in [DATA_DIR, MODEL_DIR, VIDEO_TEMPLATE_DIR, SCENE_IMAGE_DIR, VOICE_DIR, API_OUTPUT_ROOT, RUNTIME_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("VIDEO_TEMPLATE_DIR:", VIDEO_TEMPLATE_DIR)
print("SCENE_IMAGE_DIR:", SCENE_IMAGE_DIR)
print("CHECKPOINT_SRC:", CHECKPOINT_SRC)
print("API_OUTPUT_ROOT:", API_OUTPUT_ROOT)
print("REF_AUDIO_PATH:", REF_AUDIO_PATH)


In [ ]:
# ============================================================
# 3) MOUNT DRIVE + SETUP WAV2LIP ONCE
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

import shutil
import subprocess
import re
from pathlib import Path

def run_cmd(cmd, cwd=None, check=True, tail_chars=4000):
    cmd = [str(x) for x in cmd]
    print("\n$ " + " ".join(cmd))
    result = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if result.stdout:
        print(result.stdout[-tail_chars:])
    if check and result.returncode != 0:
        raise RuntimeError(
            f"Command failed with code {result.returncode}: {' '.join(cmd)}\n"
            f"Last output:\n{result.stdout[-tail_chars:]}"
        )
    return result

# Clone Wav2Lip nếu chưa có
if not WAV2LIP_DIR.exists():
    run_cmd(["git", "clone", "https://github.com/Rudrabha/Wav2Lip.git", str(WAV2LIP_DIR)])
else:
    print("Wav2Lip already exists:", WAV2LIP_DIR)

# Cài deps legacy của Wav2Lip, sau đó khóa numpy/librosa để tránh lỗi runtime phổ biến.
run_cmd(["bash", "-lc", f"cd {WAV2LIP_DIR} && pip install -q -r requirements.txt"])
run_cmd(["bash", "-lc", 'pip install -q "numpy<2.0" librosa==0.9.2 opencv-python-headless tqdm scipy'])

# Tải S3FD face detector
s3fd_path = WAV2LIP_DIR / "face_detection" / "detection" / "sfd" / "s3fd.pth"
s3fd_path.parent.mkdir(parents=True, exist_ok=True)
if not s3fd_path.exists():
    run_cmd([
        "wget", "-q",
        "https://www.adrianbulat.com/downloads/python-fan/s3fd-619a316812.pth",
        "-O", str(s3fd_path)
    ])
else:
    print("s3fd exists:", s3fd_path)

# Copy checkpoint từ Drive
CHECKPOINT_DST.parent.mkdir(parents=True, exist_ok=True)
if CHECKPOINT_SRC.exists():
    shutil.copy2(CHECKPOINT_SRC, CHECKPOINT_DST)
    print("Copied checkpoint:", CHECKPOINT_DST)
elif CHECKPOINT_DST.exists():
    print("Checkpoint already exists:", CHECKPOINT_DST)
else:
    raise FileNotFoundError(
        f"Không thấy checkpoint. Cần đặt model tại: {CHECKPOINT_SRC}"
    )

# Patch torch.load cho PyTorch 2.6+
inference_path = WAV2LIP_DIR / "inference.py"
text = inference_path.read_text(encoding="utf-8")

backup_path = WAV2LIP_DIR / "inference.py.original.bak"
if not backup_path.exists():
    backup_path.write_text(text, encoding="utf-8")

pattern = r"def _load\(checkpoint_path\):[\s\S]*?\ndef load_model\(path\):"
replacement = """def _load(checkpoint_path):
    if device == 'cuda':
        checkpoint = torch.load(checkpoint_path, weights_only=False)
    else:
        checkpoint = torch.load(
            checkpoint_path,
            map_location=lambda storage, loc: storage,
            weights_only=False
        )
    return checkpoint


def load_model(path):"""

if re.search(pattern, text):
    text = re.sub(pattern, replacement, text)
else:
    # Fallback minimal patch nếu inference.py khác format.
    text = text.replace(
        "torch.load(checkpoint_path)",
        "torch.load(checkpoint_path, weights_only=False)"
    )
    text = text.replace(
        "torch.load(checkpoint_path, map_location=lambda storage, loc: storage)",
        "torch.load(checkpoint_path, map_location=lambda storage, loc: storage, weights_only=False)"
    )

inference_path.write_text(text, encoding="utf-8")
print("Wav2Lip setup done.")


In [ ]:
# ============================================================
# 4) LOAD OMNIVOICE ONCE
# ============================================================
import os
import random
import json
import traceback
from datetime import datetime, UTC
from pathlib import Path

import torch
import soundfile as sf
import pandas as pd
from tqdm.auto import tqdm

os.environ.setdefault("PYTHONHASHSEED", str(SEED))
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Font hỗ trợ tiếng Việt cho overlay
FONT_PATH = "/usr/share/fonts/truetype/noto/NotoSans-Regular.ttf"
if not Path(FONT_PATH).exists():
    candidates = list(Path("/usr/share/fonts").rglob("NotoSans-Regular.ttf"))
    FONT_PATH = str(candidates[0]) if candidates else "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"

# Reference voice
if not Path(REF_AUDIO_PATH).exists():
    try:
        from google.colab import files
        print("Không thấy REF_AUDIO_PATH trên Drive.")
        print("Upload 1 file reference audio tiếng Việt, 3-10 giây, một người nói:")
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("Clone voice cần reference audio.")
        uploaded_name = next(iter(uploaded.keys()))
        REF_AUDIO_PATH = Path("/content") / uploaded_name
    except ImportError:
        raise FileNotFoundError(f"Không thấy reference audio: {REF_AUDIO_PATH}")

REF_AUDIO_PATH = str(REF_AUDIO_PATH)
print("LOCKED REF_AUDIO_PATH:", REF_AUDIO_PATH)
print("FONT_PATH:", FONT_PATH)

from omnivoice import OmniVoice

if torch.cuda.is_available():
    device_map = "cuda:0"
    dtype = torch.float16
else:
    device_map = "cpu"
    dtype = torch.float32

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
print("device_map:", device_map)
print("dtype:", dtype)

omni_model = OmniVoice.from_pretrained(
    MODEL_ID,
    device_map=device_map,
    dtype=dtype,
    load_asr=True,
)

print("OmniVoice loaded.")


In [ ]:
# ============================================================
# 5) CORE HELPERS: TTS, FFMPEG, WAV2LIP, ASSET RESOLUTION
# ============================================================
import csv
import json
import shutil
import subprocess
import uuid
import re
from pathlib import Path
from datetime import datetime, UTC

import numpy as np

IMAGE_EXTS = [".png", ".jpg", ".jpeg", ".webp"]
VIDEO_EXTS = [".mp4", ".mov", ".mkv", ".webm"]

def safe_filename(name, fallback="scene"):
    value = str(name or fallback).strip()
    value = re.sub(r"[^a-zA-Z0-9_.-]+", "_", value)
    return value or fallback

def normalize_text(text):
    value = str(text or "").strip()
    return re.sub(r"\s+", " ", value)

def now_id(prefix="job"):
    return f"{prefix}_{datetime.now(UTC).strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:6]}"

def run(cmd, cwd=None, check=True, tail_chars=5000):
    cmd = [str(x) for x in cmd]
    print("\n$ " + " ".join(cmd))
    result = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if result.stdout:
        print(result.stdout[-tail_chars:])
    if check and result.returncode != 0:
        raise RuntimeError(
            f"Command failed with code {result.returncode}: {' '.join(cmd)}\n"
            f"Last output:\n{result.stdout[-tail_chars:]}"
        )
    return result

def ffprobe_duration(path: Path) -> float:
    result = subprocess.check_output([
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1",
        str(path)
    ])
    return float(result.decode("utf-8").strip())

def has_audio(path: Path) -> bool:
    result = subprocess.check_output([
        "ffprobe", "-v", "error",
        "-select_streams", "a",
        "-show_entries", "stream=codec_type",
        "-of", "csv=p=0",
        str(path)
    ])
    return bool(result.decode("utf-8").strip())

def call_omnivoice_generate(text, ref_audio_path=None, ref_text=None, duration=None):
    ref_audio_path = str(ref_audio_path or REF_AUDIO_PATH)
    kwargs = {
        "text": text,
        "language_id": LANGUAGE_ID,
        "ref_audio": ref_audio_path,
        "num_step": NUM_STEP,
        "speed": SPEED,
    }
    if ref_text:
        kwargs["ref_text"] = ref_text
    if duration is not None:
        kwargs["duration"] = duration

    try:
        return omni_model.generate(**kwargs)
    except TypeError as exc:
        msg = str(exc)
        fallback_keys = ("language_id", "num_step", "speed", "duration")
        if not any(key in msg for key in fallback_keys) and "unexpected keyword" not in msg:
            raise
        print("Fallback: installed OmniVoice does not accept all generation kwargs.")
        minimal_kwargs = {"text": text, "ref_audio": ref_audio_path}
        if ref_text:
            minimal_kwargs["ref_text"] = ref_text
        return omni_model.generate(**minimal_kwargs)

def unpack_audio(audio):
    sample_rate = SAMPLE_RATE
    wav = audio

    if isinstance(audio, dict):
        wav = audio.get("audio") or audio.get("wav") or audio.get("waveform")
        sample_rate = int(audio.get("sample_rate") or audio.get("sampling_rate") or sample_rate)
    elif isinstance(audio, tuple):
        wav = audio[0]
        if len(audio) > 1 and isinstance(audio[1], int):
            sample_rate = audio[1]
    elif isinstance(audio, list):
        wav = audio[0]

    if hasattr(wav, "detach"):
        wav = wav.detach().cpu().float().numpy()

    wav = np.asarray(wav)
    return wav, sample_rate

def generate_tts_to_file(text, out_wav: Path, ref_audio_path=None, ref_text=None, duration=None):
    text = normalize_text(text)
    if not text:
        raise ValueError("Missing text/voiceover for TTS.")

    out_wav.parent.mkdir(parents=True, exist_ok=True)
    audio = call_omnivoice_generate(
        text=text,
        ref_audio_path=ref_audio_path,
        ref_text=ref_text or REF_TEXT,
        duration=duration if duration is not None else DEFAULT_DURATION,
    )
    wav, sample_rate = unpack_audio(audio)
    sf.write(out_wav, wav, sample_rate)
    return out_wav

def infer_needs_lipsync(scene):
    if "needs_lipsync" in scene:
        return bool(scene.get("needs_lipsync"))
    scene_type = str(scene.get("scene_type", "")).upper()
    non_lipsync_types = {"PRODUCT_CLOSEUP", "PRODUCT", "B_ROLL", "IMAGE", "SCREEN", "CUTAWAY"}
    return scene_type not in non_lipsync_types

def candidate_video_paths(scene):
    scene_id = scene.get("scene_id")
    scene_type = scene.get("scene_type", "")
    candidates = []

    # request override
    if scene.get("visual_path"):
        candidates.append(Path(scene["visual_path"]))

    for ext in VIDEO_EXTS:
        candidates.append(VIDEO_TEMPLATE_DIR / f"{scene_id}{ext}")

    if scene_type:
        for ext in VIDEO_EXTS:
            candidates.append(VIDEO_TEMPLATE_DIR / f"{scene_type}{ext}")

    return candidates

def candidate_image_paths(scene):
    scene_id = scene.get("scene_id")
    scene_type = scene.get("scene_type", "")
    candidates = []

    if scene.get("image_path"):
        candidates.append(Path(scene["image_path"]))

    for ext in IMAGE_EXTS:
        candidates.append(SCENE_IMAGE_DIR / f"{scene_id}{ext}")

    if scene_type:
        for ext in IMAGE_EXTS:
            candidates.append(SCENE_IMAGE_DIR / f"{scene_type}{ext}")

    return candidates

def find_visual_asset(scene):
    needs_lipsync = infer_needs_lipsync(scene)
    scene_type = scene.get("scene_type", "")

    for p in candidate_video_paths(scene):
        if p and Path(p).exists():
            return {"path": Path(p), "kind": "video"}

    if needs_lipsync:
        for name in ["HOST_TALK.mp4", "HOOK.mp4", "FAQ_ANSWER.mp4", "CTA.mp4"]:
            p = VIDEO_TEMPLATE_DIR / name
            if p.exists():
                return {"path": p, "kind": "video"}

    for p in candidate_image_paths(scene):
        if p and Path(p).exists():
            if needs_lipsync and not ALLOW_STATIC_IMAGE_FOR_LIPSYNC:
                continue
            return {"path": Path(p), "kind": "image"}

    if (not needs_lipsync) or str(scene_type).upper() == "PRODUCT_CLOSEUP":
        for name in ["model.png", "product.png", "PRODUCT_CLOSEUP.png"]:
            p = SCENE_IMAGE_DIR / name
            if p.exists():
                return {"path": p, "kind": "image"}

    return None

def normalize_video_to_audio(template_video: Path, audio_path: Path, out_path: Path):
    dur = ffprobe_duration(audio_path)
    vf = (
        f"scale={TARGET_W}:{TARGET_H}:force_original_aspect_ratio=increase,"
        f"crop={TARGET_W}:{TARGET_H},"
        f"fps={FPS},"
        "format=yuv420p"
    )

    run([
        "ffmpeg", "-y",
        "-stream_loop", "-1",
        "-i", template_video,
        "-t", f"{dur:.3f}",
        "-an",
        "-vf", vf,
        "-c:v", "libx264",
        "-preset", PRESET,
        "-crf", str(CRF),
        "-pix_fmt", "yuv420p",
        out_path
    ])
    return out_path

def image_to_motion_video(image_path: Path, audio_path: Path, out_path: Path):
    dur = ffprobe_duration(audio_path)
    frames = max(1, int(dur * FPS))
    vf = (
        f"scale={TARGET_W}:{TARGET_H}:force_original_aspect_ratio=increase,"
        f"crop={TARGET_W}:{TARGET_H},"
        f"zoompan=z='min(zoom+0.0008,1.06)':d={frames}:s={TARGET_W}x{TARGET_H}:fps={FPS},"
        "format=yuv420p"
    )

    run([
        "ffmpeg", "-y",
        "-loop", "1",
        "-i", image_path,
        "-t", f"{dur:.3f}",
        "-vf", vf,
        "-c:v", "libx264",
        "-preset", PRESET,
        "-crf", str(CRF),
        "-pix_fmt", "yuv420p",
        out_path
    ])
    return out_path

def mux_audio(video_path: Path, audio_path: Path, out_path: Path):
    run([
        "ffmpeg", "-y",
        "-i", video_path,
        "-i", audio_path,
        "-map", "0:v:0",
        "-map", "1:a:0",
        "-c:v", "libx264",
        "-preset", PRESET,
        "-crf", str(CRF),
        "-c:a", "aac",
        "-ar", "44100",
        "-b:a", "192k",
        "-shortest",
        "-pix_fmt", "yuv420p",
        out_path
    ])
    return out_path

def add_overlay_text(video_path: Path, text: str, out_path: Path):
    if not text:
        shutil.copy2(video_path, out_path)
        return out_path

    overlay_runtime_dir = Path("/content/overlay_texts")
    overlay_runtime_dir.mkdir(parents=True, exist_ok=True)

    text_file = overlay_runtime_dir / f"{out_path.stem}_overlay.txt"
    text = str(text).replace("\r\n", "\n").replace("\r", "\n").strip()
    text_file.write_text(text, encoding="utf-8")

    draw = (
        f"drawtext=fontfile='{FONT_PATH}':"
        f"textfile='{text_file}':"
        "fontcolor=white:"
        "fontsize=42:"
        "line_spacing=10:"
        "box=1:"
        "boxcolor=black@0.58:"
        "boxborderw=24:"
        "x=(w-text_w)/2:"
        "y=h-text_h-180"
    )

    run([
        "ffmpeg", "-y",
        "-i", video_path,
        "-vf", draw,
        "-map", "0:v:0",
        "-map", "0:a:0?",
        "-c:v", "libx264",
        "-preset", PRESET,
        "-crf", str(CRF),
        "-c:a", "aac",
        "-ar", "44100",
        "-b:a", "192k",
        "-pix_fmt", "yuv420p",
        out_path
    ])
    return out_path

def run_wav2lip(face_video: Path, audio_path: Path, out_path: Path, scene_id: str):
    RUNTIME_DIR.mkdir(parents=True, exist_ok=True)

    local_face = RUNTIME_DIR / f"{scene_id}_face.mp4"
    local_audio = RUNTIME_DIR / f"{scene_id}_audio.wav"
    local_temp = RUNTIME_DIR / f"{scene_id}_wav2lip_temp.mp4"

    shutil.copy2(face_video, local_face)
    shutil.copy2(audio_path, local_audio)

    result_path = WAV2LIP_DIR / "results" / "result_voice.mp4"
    if result_path.exists():
        result_path.unlink()

    run([
        "python", "inference.py",
        "--checkpoint_path", CHECKPOINT_DST,
        "--face", local_face,
        "--audio", local_audio,
        "--pads", *PADS,
        "--resize_factor", RESIZE_FACTOR,
        "--wav2lip_batch_size", WAV2LIP_BATCH_SIZE,
    ], cwd=WAV2LIP_DIR)

    if not result_path.exists():
        raise RuntimeError(f"Wav2Lip không tạo result_voice.mp4 cho {scene_id}")

    shutil.copy2(result_path, local_temp)
    mux_audio(local_temp, local_audio, out_path)
    return out_path

def concat_videos(video_paths, out_path: Path, work_dir: Path):
    if not video_paths:
        raise ValueError("No videos to concat.")

    concat_file = work_dir / f"{out_path.stem}_concat.txt"
    concat_file.write_text(
        "\n".join([f"file '{Path(p).resolve()}'" for p in video_paths]),
        encoding="utf-8"
    )

    run([
        "ffmpeg", "-y",
        "-f", "concat",
        "-safe", "0",
        "-i", concat_file,
        "-c:v", "libx264",
        "-preset", PRESET,
        "-crf", str(CRF),
        "-c:a", "aac",
        "-ar", "44100",
        "-b:a", "192k",
        "-pix_fmt", "yuv420p",
        out_path
    ])

    if not has_audio(out_path):
        raise RuntimeError(f"Concat output không có audio: {out_path}")

    return out_path

print("Core helpers ready.")


In [ ]:
# ============================================================
# 6) RENDER ONE SCENE / ONE CHUNK
# ============================================================
PUBLIC_URL = None

def get_job_dirs(job_id):
    job_id = safe_filename(job_id, fallback=now_id("job"))
    job_root = API_OUTPUT_ROOT / job_id
    dirs = {
        "root": job_root,
        "audio": job_root / "audio",
        "text": job_root / "text",
        "work": job_root / "_work",
        "scenes": job_root / "scenes",
        "final": job_root / "final",
        "reports": job_root / "reports",
    }
    for p in dirs.values():
        p.mkdir(parents=True, exist_ok=True)
    return job_id, dirs

def file_url(job_id, folder, filename):
    if PUBLIC_URL:
        return f"{PUBLIC_URL}/files/{job_id}/{folder}/{filename}"
    return f"/files/{job_id}/{folder}/{filename}"

def render_scene_chunk(scene, job_id=None, sequence_index=None):
    """
    scene payload fields:
    - scene_id: "S001"
    - text or voiceover: nội dung để OmniVoice sinh WAV
    - scene_type: HOST_TALK / PRODUCT_CLOSEUP / CTA / ...
    - needs_lipsync: true/false
    - overlay_text: optional
    - visual_path: optional video override
    - image_path: optional image override
    - audio_path: optional existing wav override, nếu muốn skip TTS
    """
    job_id = job_id or scene.get("job_id") or now_id("job")
    job_id, dirs = get_job_dirs(job_id)

    scene_id = safe_filename(scene.get("scene_id") or f"S{int(sequence_index or 1):03d}")
    scene = {**scene, "scene_id": scene_id}

    text = normalize_text(scene.get("text") or scene.get("voiceover") or scene.get("dialogue") or "")
    overlay_text = scene.get("overlay_text") or scene.get("caption") or ""
    needs_lipsync = infer_needs_lipsync(scene)

    audio_path = dirs["audio"] / f"{scene_id}.wav"
    text_path = dirs["text"] / f"{scene_id}.txt"
    base_video = dirs["work"] / f"{scene_id}_base.mp4"
    raw_scene = dirs["work"] / f"{scene_id}_raw.mp4"
    final_scene = dirs["scenes"] / f"{scene_id}.mp4"

    started_at = datetime.now(UTC).isoformat().replace("+00:00", "Z")

    # 1) Audio: dùng audio_path nếu request đưa sẵn, nếu không thì sinh bằng OmniVoice
    request_audio_path = scene.get("audio_path")
    if request_audio_path and Path(request_audio_path).exists():
        shutil.copy2(request_audio_path, audio_path)
        audio_source = "provided_audio_path"
    else:
        text_path.write_text(text, encoding="utf-8")
        duration = scene.get("duration")
        if duration is None and scene.get("use_scene_duration_for_tts"):
            duration = scene.get("duration_target_sec")
        generate_tts_to_file(
            text=text,
            out_wav=audio_path,
            ref_audio_path=scene.get("ref_audio_path") or REF_AUDIO_PATH,
            ref_text=scene.get("ref_text") or REF_TEXT,
            duration=duration,
        )
        audio_source = "omnivoice_tts"

    # 2) Visual asset
    visual = find_visual_asset(scene)
    if visual is None:
        raise FileNotFoundError(
            f"Không tìm thấy visual cho {scene_id}. "
            f"Cần video_templates/{scene_id}.mp4 hoặc video_templates/{scene.get('scene_type')}.mp4. "
            f"Scene non-lipsync có thể dùng scene_images/{scene_id}.png/jpg/webp."
        )

    if visual["kind"] == "video":
        normalize_video_to_audio(visual["path"], audio_path, base_video)
    elif visual["kind"] == "image":
        image_to_motion_video(visual["path"], audio_path, base_video)
    else:
        raise ValueError(f"Unsupported visual kind: {visual['kind']}")

    # 3) Lip-sync hoặc chỉ mux audio
    if needs_lipsync:
        run_wav2lip(base_video, audio_path, raw_scene, scene_id)
    else:
        mux_audio(base_video, audio_path, raw_scene)

    # 4) Overlay
    add_overlay_text(raw_scene, overlay_text, final_scene)

    if not final_scene.exists():
        raise RuntimeError(f"Missing final_scene: {final_scene}")
    if not has_audio(final_scene):
        raise RuntimeError(f"Scene output không có audio: {final_scene}")

    duration_sec = ffprobe_duration(final_scene)

    result = {
        "status": "ok",
        "job_id": job_id,
        "scene_id": scene_id,
        "sequence_index": sequence_index,
        "started_at": started_at,
        "finished_at": datetime.now(UTC).isoformat().replace("+00:00", "Z"),
        "needs_lipsync": needs_lipsync,
        "audio_source": audio_source,
        "audio_path": str(audio_path),
        "audio_url": file_url(job_id, "audio", audio_path.name),
        "visual_path": str(visual["path"]),
        "visual_kind": visual["kind"],
        "video_path": str(final_scene),
        "video_url": file_url(job_id, "scenes", final_scene.name),
        "duration_sec": duration_sec,
    }

    manifest_path = dirs["reports"] / "chunks_manifest.jsonl"
    with open(manifest_path, "a", encoding="utf-8") as f:
        f.write(json.dumps(result, ensure_ascii=False) + "\n")

    return result

def concat_job(job_id, scene_results):
    job_id, dirs = get_job_dirs(job_id)
    paths = [Path(r["video_path"]) for r in scene_results if r.get("status") == "ok"]
    out_path = dirs["final"] / f"{job_id}_FULL_LOOP.mp4"
    concat_videos(paths, out_path, dirs["work"])
    result = {
        "status": "ok",
        "job_id": job_id,
        "final_path": str(out_path),
        "final_url": file_url(job_id, "final", out_path.name),
        "scene_count": len(paths),
        "duration_sec": ffprobe_duration(out_path),
    }
    (dirs["reports"] / "final_manifest.json").write_text(
        json.dumps(result, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )
    return result

print("Render functions ready.")


In [ ]:
# ============================================================
# 7) FLASK API + NGROK
# ============================================================
import json
import threading
import traceback
import getpass

import nest_asyncio
nest_asyncio.apply()

from flask import Flask, request, jsonify, Response, stream_with_context, send_from_directory
from flask_cors import CORS
from pyngrok import ngrok

app = Flask(__name__)
CORS(app)

RENDER_LOCK = threading.Lock()

@app.get("/")
def index():
    return jsonify({
        "service": "cocoon-omnivoice-wav2lip-ngrok",
        "status": "ok",
        "endpoints": {
            "health": "GET /health",
            "render_one_chunk": "POST /render_chunk",
            "render_job_sync": "POST /render_job",
            "render_job_stream_sse": "POST /render_job_stream",
            "files": "GET /files/<job_id>/<folder>/<filename>",
        },
        "sample_scene": {
            "job_id": "demo_live_001",
            "scene_id": "S001",
            "scene_type": "HOST_TALK",
            "needs_lipsync": True,
            "text": "Dạ mọi người ơi, hôm nay em lên live để giới thiệu một sản phẩm rất đáng thử.",
            "overlay_text": "Ưu đãi trong live hôm nay"
        }
    })

@app.get("/health")
def health():
    return jsonify({
        "status": "ok",
        "cuda": torch.cuda.is_available(),
        "project_dir": str(PROJECT_DIR),
        "api_output_root": str(API_OUTPUT_ROOT),
        "public_url": PUBLIC_URL,
    })

@app.get("/files/<job_id>/<folder>/<filename>")
def serve_file(job_id, folder, filename):
    allowed = {"audio", "scenes", "final", "reports"}
    if folder not in allowed:
        return jsonify({"status": "error", "error": "folder_not_allowed"}), 400

    base = API_OUTPUT_ROOT / safe_filename(job_id) / folder
    return send_from_directory(base, filename, as_attachment=False)

@app.post("/render_chunk")
def render_chunk_route():
    payload = request.get_json(force=True, silent=False)
    if not isinstance(payload, dict):
        return jsonify({"status": "error", "error": "JSON body must be an object."}), 400

    try:
        with RENDER_LOCK:
            result = render_scene_chunk(payload, job_id=payload.get("job_id"), sequence_index=payload.get("sequence_index"))
        return jsonify(result)
    except Exception as exc:
        return jsonify({
            "status": "error",
            "error": repr(exc),
            "traceback": traceback.format_exc(),
        }), 500

@app.post("/render_job")
def render_job_route():
    payload = request.get_json(force=True, silent=False)
    scenes = payload.get("scenes", [])
    if not scenes:
        return jsonify({"status": "error", "error": "Missing non-empty scenes array."}), 400

    job_id = safe_filename(payload.get("job_id") or now_id("job"))

    try:
        results = []
        with RENDER_LOCK:
            for i, scene in enumerate(scenes, start=1):
                scene = {**scene, "job_id": job_id}
                results.append(render_scene_chunk(scene, job_id=job_id, sequence_index=i))

            final = None
            if payload.get("concat_final", True):
                final = concat_job(job_id, results)

        return jsonify({
            "status": "ok",
            "job_id": job_id,
            "chunks": results,
            "final": final,
        })
    except Exception as exc:
        return jsonify({
            "status": "error",
            "job_id": job_id,
            "error": repr(exc),
            "traceback": traceback.format_exc(),
            "partial_chunks": results if "results" in locals() else [],
        }), 500

def sse_event(data):
    return "data: " + json.dumps(data, ensure_ascii=False) + "\n\n"

@app.post("/render_job_stream")
def render_job_stream_route():
    payload = request.get_json(force=True, silent=False)
    scenes = payload.get("scenes", [])
    if not scenes:
        return jsonify({"status": "error", "error": "Missing non-empty scenes array."}), 400

    job_id = safe_filename(payload.get("job_id") or now_id("job"))

    def generate():
        results = []
        yield sse_event({
            "type": "job_start",
            "status": "ok",
            "job_id": job_id,
            "scene_count": len(scenes),
        })

        with RENDER_LOCK:
            for i, scene in enumerate(scenes, start=1):
                scene_id = safe_filename(scene.get("scene_id") or f"S{i:03d}")
                yield sse_event({
                    "type": "scene_start",
                    "status": "ok",
                    "job_id": job_id,
                    "scene_id": scene_id,
                    "sequence_index": i,
                })

                try:
                    scene = {**scene, "scene_id": scene_id, "job_id": job_id}
                    result = render_scene_chunk(scene, job_id=job_id, sequence_index=i)
                    results.append(result)
                    yield sse_event({"type": "scene_done", **result})
                except Exception as exc:
                    error = {
                        "type": "scene_error",
                        "status": "error",
                        "job_id": job_id,
                        "scene_id": scene_id,
                        "sequence_index": i,
                        "error": repr(exc),
                        "traceback": traceback.format_exc(),
                    }
                    yield sse_event(error)
                    if payload.get("stop_on_error", True):
                        yield sse_event({
                            "type": "job_stop",
                            "status": "error",
                            "job_id": job_id,
                            "reason": "stop_on_error",
                        })
                        return

            if payload.get("concat_final", True) and results:
                try:
                    final = concat_job(job_id, results)
                    yield sse_event({"type": "final_done", **final})
                except Exception as exc:
                    yield sse_event({
                        "type": "final_error",
                        "status": "error",
                        "job_id": job_id,
                        "error": repr(exc),
                        "traceback": traceback.format_exc(),
                    })

        yield sse_event({
            "type": "job_done",
            "status": "ok",
            "job_id": job_id,
            "chunks_done": len(results),
        })

    return Response(stream_with_context(generate()), mimetype="text/event-stream")

def start_server():
    global PUBLIC_URL

    token = NGROK_AUTHTOKEN
    if not token:
        token = getpass.getpass("Paste NGROK_AUTHTOKEN: ").strip()

    if token:
        ngrok.set_auth_token(token)

    try:
        ngrok.kill()
    except Exception:
        pass

    tunnel = ngrok.connect(PORT)
    PUBLIC_URL = tunnel.public_url.rstrip("/")
    print("NGROK PUBLIC_URL:", PUBLIC_URL)

    thread = threading.Thread(
        target=lambda: app.run(host=HOST, port=PORT, debug=False, use_reloader=False),
        daemon=True,
    )
    thread.start()
    print("Flask server started.")
    print("Health:", f"{PUBLIC_URL}/health")
    print("Render chunk:", f"{PUBLIC_URL}/render_chunk")
    print("Render job stream:", f"{PUBLIC_URL}/render_job_stream")

start_server()


In [ ]:
# ============================================================
# 8) TEST REQUEST: ONE CHUNK
# ============================================================
import requests
import json

sample_chunk = {
    "job_id": "demo_live_001",
    "scene_id": "S001",
    "scene_type": "HOST_TALK",
    "needs_lipsync": True,
    "text": "Dạ mọi người ơi, hôm nay em lên live để giới thiệu một sản phẩm rất đáng thử.",
    "overlay_text": "Ưu đãi trong live hôm nay"
}

r = requests.post(f"{PUBLIC_URL}/render_chunk", json=sample_chunk, timeout=None)
print("status:", r.status_code)
print(json.dumps(r.json(), ensure_ascii=False, indent=2))


In [ ]:
# ============================================================
# 9) TEST REQUEST: STREAM JOB, TRẢ VIDEO URL THEO TỪNG CHUNK
# ============================================================
import requests
import json

sample_job = {
    "job_id": "demo_stream_001",
    "concat_final": True,
    "stop_on_error": True,
    "scenes": [
        {
            "scene_id": "S001",
            "scene_type": "HOST_TALK",
            "needs_lipsync": True,
            "text": "Dạ mọi người ơi, hôm nay em lên live để giới thiệu sản phẩm Cocoon rất đáng thử.",
            "overlay_text": "Mở đầu livestream"
        },
        {
            "scene_id": "S002",
            "scene_type": "PRODUCT_CLOSEUP",
            "needs_lipsync": False,
            "text": "Đây là phần cận cảnh sản phẩm, mọi người nhìn texture rất rõ và dễ quan sát.",
            "overlay_text": "Cận cảnh sản phẩm"
        },
        {
            "scene_id": "S003",
            "scene_type": "CTA",
            "needs_lipsync": True,
            "text": "Nếu mọi người muốn đặt hàng thì bấm vào giỏ hàng ngay trong live giúp em nha.",
            "overlay_text": "Bấm giỏ hàng để nhận ưu đãi"
        },
    ],
}

with requests.post(f"{PUBLIC_URL}/render_job_stream", json=sample_job, stream=True, timeout=None) as resp:
    print("status:", resp.status_code)
    for raw_line in resp.iter_lines(decode_unicode=True):
        if not raw_line:
            continue
        if raw_line.startswith("data: "):
            event = json.loads(raw_line[len("data: "):])
            print(json.dumps(event, ensure_ascii=False, indent=2))


## Request format nhanh

Gọi endpoint một chunk:

```bash
curl -X POST "$PUBLIC_URL/render_chunk" \
  -H "Content-Type: application/json" \
  -d '{
    "job_id": "live_001",
    "scene_id": "S001",
    "scene_type": "HOST_TALK",
    "needs_lipsync": true,
    "text": "Dạ mọi người ơi, hôm nay em giới thiệu sản phẩm này...",
    "overlay_text": "Ưu đãi trong live"
  }'
```

Gọi endpoint stream nhiều chunk:

```bash
curl -N -X POST "$PUBLIC_URL/render_job_stream" \
  -H "Content-Type: application/json" \
  -d @job.json
```

Mỗi event `scene_done` sẽ có `video_url`. Event `final_done` sẽ có `final_url`.
